# Heterogeneous Graph Modeling for Anti-Money Laundering
## A Reproducible Study Using Elliptic++ Dataset

**Authors:** Abdullah Almekhyal, Yousef Joudeh  
**Affiliation:** Computer Science Department, College of Science, Kuwait University

---

This notebook preprocesses the Elliptic++ dataset and constructs a heterogeneous graph
for AML detection, following the methodology of Johannessen & Jullum (2025).

**Anchor Paper:** *Finding Money Launderers Using Heterogeneous Graph Neural Networks*  
**Dataset:** Elliptic++ (Elmougy & Liu, KDD 2023)

## 1. Environment Setup

In [ ]:
# Install required libraries
!pip install torch-geometric
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.1.0+cu121.html
!pip install scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from collections import Counter

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Mount Google Drive & Load Data

Upload the Elliptic++ dataset to your Google Drive first.  
Expected folder structure:
```
MyDrive/
  EllipticPlusPlus/
    Actors Dataset/
      wallets_features.csv
      wallets_classes.csv
      AddrAddr_edgelist.csv
      AddrTx_edgelist.csv
      TxAddr_edgelist.csv
    Transactions Dataset/
      txs_features.csv
      txs_classes.csv
      txs_edgelist.csv
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# UPDATE THIS PATH to match your Google Drive folder structure
# ============================================================
BASE_PATH = '/content/drive/MyDrive/EllipticPlusPlus'

ACTORS_PATH = f'{BASE_PATH}/Actors Dataset'
TXS_PATH = f'{BASE_PATH}/Transactions Dataset'

In [ ]:
# ---- Load Transaction Data ----
print('Loading transaction features...')
txs_features = pd.read_csv(f'{TXS_PATH}/txs_features.csv', header=None)
print(f'  Shape: {txs_features.shape}')

print('Loading transaction classes...')
txs_classes = pd.read_csv(f'{TXS_PATH}/txs_classes.csv')
print(f'  Shape: {txs_classes.shape}')

print('Loading transaction edgelist...')
txs_edgelist = pd.read_csv(f'{TXS_PATH}/txs_edgelist.csv')
print(f'  Shape: {txs_edgelist.shape}')

print('\n---- Load Actor (Wallet) Data ----')
print('Loading wallet features...')
wallets_features = pd.read_csv(f'{ACTORS_PATH}/wallets_features.csv')
print(f'  Shape: {wallets_features.shape}')

print('Loading wallet classes...')
wallets_classes = pd.read_csv(f'{ACTORS_PATH}/wallets_classes.csv')
print(f'  Shape: {wallets_classes.shape}')

print('Loading AddrTx edgelist (wallet -> transaction)...')
addr_tx_edges = pd.read_csv(f'{ACTORS_PATH}/AddrTx_edgelist.csv')
print(f'  Shape: {addr_tx_edges.shape}')

print('Loading TxAddr edgelist (transaction -> wallet)...')
tx_addr_edges = pd.read_csv(f'{ACTORS_PATH}/TxAddr_edgelist.csv')
print(f'  Shape: {tx_addr_edges.shape}')

print('Loading AddrAddr edgelist (wallet <-> wallet)...')
addr_addr_edges = pd.read_csv(f'{ACTORS_PATH}/AddrAddr_edgelist.csv')
print(f'  Shape: {addr_addr_edges.shape}')

print('\nAll files loaded successfully!')

## 3. Data Exploration

In [ ]:
# Inspect transaction features
print('=== TRANSACTION FEATURES ===')
print(f'Shape: {txs_features.shape}')
print(f'Columns: {txs_features.columns.tolist()[:10]}... (first 10)')
print(f'\nFirst 3 rows:')
txs_features.head(3)

In [ ]:
# Inspect transaction classes
print('=== TRANSACTION CLASSES ===')
print(txs_classes.head())
print(f'\nColumn names: {txs_classes.columns.tolist()}')
print(f'\nClass distribution:')
print(txs_classes.iloc[:, -1].value_counts())

In [ ]:
# Inspect wallet features
print('=== WALLET FEATURES ===')
print(f'Shape: {wallets_features.shape}')
print(f'Columns: {wallets_features.columns.tolist()}')
print(f'\nFirst 3 rows:')
wallets_features.head(3)

In [ ]:
# Inspect wallet classes
print('=== WALLET CLASSES ===')
print(wallets_classes.head())
print(f'\nColumn names: {wallets_classes.columns.tolist()}')
print(f'\nClass distribution:')
print(wallets_classes.iloc[:, -1].value_counts())

In [ ]:
# Inspect edge files
print('=== EDGE FILES ===')
print(f'\nAddrTx (wallet -> txn) columns: {addr_tx_edges.columns.tolist()}')
print(addr_tx_edges.head())

print(f'\nTxAddr (txn -> wallet) columns: {tx_addr_edges.columns.tolist()}')
print(tx_addr_edges.head())

print(f'\nAddrAddr (wallet <-> wallet) columns: {addr_addr_edges.columns.tolist()}')
print(addr_addr_edges.head())

## 4. Data Cleaning & Preprocessing

In [ ]:
# ============================================================
# 4.1 Identify column names
# ============================================================
# Transaction features: column 0 = txn ID, column 1 = timestep,
#                       columns 2-183 = features
# Wallet features: first column = address, rest = features

# --- Transaction node IDs and features ---
txn_ids = txs_features.iloc[:, 0].values          # transaction IDs
txn_timesteps = txs_features.iloc[:, 1].values     # timestep (1-49)
txn_feat_raw = txs_features.iloc[:, 2:].values     # actual features

print(f'Transaction nodes: {len(txn_ids)}')
print(f'Transaction features per node: {txn_feat_raw.shape[1]}')
print(f'Timesteps range: {txn_timesteps.min()} to {txn_timesteps.max()}')

# --- Wallet node IDs and features ---
wallet_id_col = wallets_features.columns[0]  # address column
wallet_ids = wallets_features[wallet_id_col].values
wallet_feat_cols = wallets_features.columns[1:]  # feature columns
wallet_feat_raw = wallets_features[wallet_feat_cols].values

print(f'\nWallet nodes: {len(wallet_ids)}')
print(f'Wallet features per node: {wallet_feat_raw.shape[1]}')

In [ ]:
# ============================================================
# 4.2 Process labels
# ============================================================
# Transaction labels: 1 = illicit, 2 = licit, unknown/3 = unlabeled
# Wallet labels: same convention
#
# We remap: illicit(1) -> 1, licit(2) -> 0, unknown -> -1
# This matches the anchor paper: class 1 = suspicious, class 0 = regular

def process_labels(classes_df):
    """Remap labels: illicit=1, licit=0, unknown=-1"""
    id_col = classes_df.columns[0]
    label_col = classes_df.columns[-1]

    ids = classes_df[id_col].values
    raw_labels = classes_df[label_col].values

    labels = np.full(len(raw_labels), -1, dtype=np.int64)
    # Handle both string and int labels
    for i, lbl in enumerate(raw_labels):
        lbl_str = str(lbl).strip().lower()
        if lbl_str == '1' or lbl_str == 'illicit':
            labels[i] = 1   # illicit / suspicious
        elif lbl_str == '2' or lbl_str == 'licit':
            labels[i] = 0   # licit / regular
        else:
            labels[i] = -1  # unknown

    return ids, labels

txn_label_ids, txn_labels = process_labels(txs_classes)
wallet_label_ids, wallet_labels = process_labels(wallets_classes)

print('Transaction label distribution:')
print(f'  Illicit (1): {(txn_labels == 1).sum()}')
print(f'  Licit   (0): {(txn_labels == 0).sum()}')
print(f'  Unknown(-1): {(txn_labels == -1).sum()}')

print(f'\nWallet label distribution:')
print(f'  Illicit (1): {(wallet_labels == 1).sum()}')
print(f'  Licit   (0): {(wallet_labels == 0).sum()}')
print(f'  Unknown(-1): {(wallet_labels == -1).sum()}')

In [ ]:
# ============================================================
# 4.3 Visualize class imbalance
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Transaction labels
txn_labeled_mask = txn_labels >= 0
txn_labeled = txn_labels[txn_labeled_mask]
counts_txn = [np.sum(txn_labeled == 0), np.sum(txn_labeled == 1)]
axes[0].bar(['Licit', 'Illicit'], counts_txn, color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Transaction Labels (labeled only)')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts_txn):
    axes[0].text(i, v + 200, str(v), ha='center', fontweight='bold')

# Wallet labels
wal_labeled_mask = wallet_labels >= 0
wal_labeled = wallet_labels[wal_labeled_mask]
counts_wal = [np.sum(wal_labeled == 0), np.sum(wal_labeled == 1)]
axes[1].bar(['Licit', 'Illicit'], counts_wal, color=['#2ecc71', '#e74c3c'])
axes[1].set_title('Wallet Labels (labeled only)')
axes[1].set_ylabel('Count')
for i, v in enumerate(counts_wal):
    axes[1].text(i, v + 500, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

# Print imbalance ratios
print(f'Transaction illicit ratio: {counts_txn[1]/(counts_txn[0]+counts_txn[1])*100:.2f}%')
print(f'Wallet illicit ratio: {counts_wal[1]/(counts_wal[0]+counts_wal[1])*100:.2f}%')

## 5. Build Node ID Mappings

PyTorch Geometric requires contiguous integer indices starting from 0.  
We create mappings for both wallet addresses and transaction IDs.

In [ ]:
# ============================================================
# 5.1 Create ID -> index mappings
# ============================================================

# Transaction ID mapping
txn_id_to_idx = {tid: i for i, tid in enumerate(txn_ids)}
print(f'Transaction ID mapping: {len(txn_id_to_idx)} nodes -> indices 0..{len(txn_id_to_idx)-1}')

# Wallet ID mapping
wallet_id_to_idx = {wid: i for i, wid in enumerate(wallet_ids)}
print(f'Wallet ID mapping: {len(wallet_id_to_idx)} nodes -> indices 0..{len(wallet_id_to_idx)-1}')

# Also create label arrays aligned with feature arrays
# (labels may not be in same order as features)
txn_label_map = dict(zip(txn_label_ids, txn_labels))
wallet_label_map = dict(zip(wallet_label_ids, wallet_labels))

# Align labels with feature ordering
txn_labels_aligned = np.array([txn_label_map.get(tid, -1) for tid in txn_ids])
wallet_labels_aligned = np.array([wallet_label_map.get(wid, -1) for wid in wallet_ids])

print(f'\nAligned transaction labels: {len(txn_labels_aligned)} (labeled: {(txn_labels_aligned >= 0).sum()})')
print(f'Aligned wallet labels: {len(wallet_labels_aligned)} (labeled: {(wallet_labels_aligned >= 0).sum()})')

## 6. Build Edge Index Tensors

In [ ]:
# ============================================================
# 6.1 Process edge files into PyG edge_index format
# ============================================================

def build_edge_index(edge_df, src_col, dst_col, src_map, dst_map):
    """
    Build edge_index tensor from edge dataframe.
    Filters out edges where src or dst is not in the mapping.
    Returns: edge_index (2, num_edges) tensor
    """
    src_ids = edge_df[src_col].values
    dst_ids = edge_df[dst_col].values

    src_indices = []
    dst_indices = []
    skipped = 0

    for s, d in zip(src_ids, dst_ids):
        if s in src_map and d in dst_map:
            src_indices.append(src_map[s])
            dst_indices.append(dst_map[d])
        else:
            skipped += 1

    edge_index = torch.tensor([src_indices, dst_indices], dtype=torch.long)
    print(f'  Built edge_index: {edge_index.shape} ({skipped} edges skipped due to missing nodes)')
    return edge_index

In [ ]:
# Identify column names from edge DataFrames
print('AddrTx columns:', addr_tx_edges.columns.tolist())
print('TxAddr columns:', tx_addr_edges.columns.tolist())
print('AddrAddr columns:', addr_addr_edges.columns.tolist())
print('TxsTxs columns:', txs_edgelist.columns.tolist())

In [ ]:
# ============================================================
# 6.2 Build all edge indices
# ============================================================
# NOTE: Update column names below based on the output of the cell above.
#       The column names depend on the CSV headers in the dataset.

# --- Edge type 1: wallet -> transaction ("sends") ---
print('Building wallet -> transaction edges ("sends"):')
addr_tx_src_col = addr_tx_edges.columns[0]  # address column
addr_tx_dst_col = addr_tx_edges.columns[1]  # transaction column
edge_wallet_to_txn = build_edge_index(
    addr_tx_edges, addr_tx_src_col, addr_tx_dst_col,
    wallet_id_to_idx, txn_id_to_idx
)

# --- Edge type 2: transaction -> wallet ("received_by") ---
print('\nBuilding transaction -> wallet edges ("received_by"):')
tx_addr_src_col = tx_addr_edges.columns[0]  # transaction column
tx_addr_dst_col = tx_addr_edges.columns[1]  # address column
edge_txn_to_wallet = build_edge_index(
    tx_addr_edges, tx_addr_src_col, tx_addr_dst_col,
    txn_id_to_idx, wallet_id_to_idx
)

# --- Edge type 3: wallet -> wallet ("interacts_with") ---
print('\nBuilding wallet -> wallet edges ("interacts_with"):')
aa_src_col = addr_addr_edges.columns[0]
aa_dst_col = addr_addr_edges.columns[1]
edge_wallet_to_wallet = build_edge_index(
    addr_addr_edges, aa_src_col, aa_dst_col,
    wallet_id_to_idx, wallet_id_to_idx
)

# --- Edge type 4: transaction -> transaction ("flows_to") ---
print('\nBuilding transaction -> transaction edges ("flows_to"):')
tt_src_col = txs_edgelist.columns[0]
tt_dst_col = txs_edgelist.columns[1]
edge_txn_to_txn = build_edge_index(
    txs_edgelist, tt_src_col, tt_dst_col,
    txn_id_to_idx, txn_id_to_idx
)

print('\n=== Edge Summary ===')
print(f'wallet  -> txn       (sends):          {edge_wallet_to_txn.shape[1]:>10,} edges')
print(f'txn     -> wallet    (received_by):     {edge_txn_to_wallet.shape[1]:>10,} edges')
print(f'wallet  -> wallet    (interacts_with):  {edge_wallet_to_wallet.shape[1]:>10,} edges')
print(f'txn     -> txn       (flows_to):        {edge_txn_to_txn.shape[1]:>10,} edges')
total_edges = (edge_wallet_to_txn.shape[1] + edge_txn_to_wallet.shape[1] +
               edge_wallet_to_wallet.shape[1] + edge_txn_to_txn.shape[1])
print(f'TOTAL:                                  {total_edges:>10,} edges')

## 7. Feature Normalization

In [ ]:
# ============================================================
# 7.1 Handle missing values and normalize features
# ============================================================

# Replace NaN/inf with 0
txn_feat_clean = np.nan_to_num(txn_feat_raw.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
wallet_feat_clean = np.nan_to_num(wallet_feat_raw.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)

print(f'Transaction features NaN count: {np.isnan(txn_feat_raw.astype(float)).sum()}')
print(f'Wallet features NaN count: {np.isnan(wallet_feat_raw.astype(float)).sum()}')

# Normalize per node type (StandardScaler)
scaler_txn = StandardScaler()
txn_feat_norm = scaler_txn.fit_transform(txn_feat_clean)

scaler_wallet = StandardScaler()
wallet_feat_norm = scaler_wallet.fit_transform(wallet_feat_clean)

print(f'\nNormalized transaction features: shape={txn_feat_norm.shape}, '
      f'mean={txn_feat_norm.mean():.4f}, std={txn_feat_norm.std():.4f}')
print(f'Normalized wallet features: shape={wallet_feat_norm.shape}, '
      f'mean={wallet_feat_norm.mean():.4f}, std={wallet_feat_norm.std():.4f}')

In [ ]:
# ============================================================
# 7.2 Convert to PyTorch tensors
# ============================================================
txn_x = torch.tensor(txn_feat_norm, dtype=torch.float32)
wallet_x = torch.tensor(wallet_feat_norm, dtype=torch.float32)

txn_y = torch.tensor(txn_labels_aligned, dtype=torch.long)
wallet_y = torch.tensor(wallet_labels_aligned, dtype=torch.long)

print(f'Transaction feature tensor: {txn_x.shape}')
print(f'Wallet feature tensor:      {wallet_x.shape}')
print(f'Transaction label tensor:   {txn_y.shape}')
print(f'Wallet label tensor:        {wallet_y.shape}')

## 8. Construct PyG HeteroData Object

This is the core data structure for heterogeneous GNNs in PyTorch Geometric.  
It matches our methodology:
- **Node types:** `wallet` (= account) and `transaction`
- **Edge types:** `sends`, `received_by`, `interacts_with`, `flows_to`

In [ ]:
from torch_geometric.data import HeteroData

data = HeteroData()

# ---- Node features ----
data['wallet'].x = wallet_x
data['transaction'].x = txn_x

# ---- Node labels ----
data['wallet'].y = wallet_y
data['transaction'].y = txn_y

# ---- Edge indices ----
data['wallet', 'sends', 'transaction'].edge_index = edge_wallet_to_txn
data['transaction', 'received_by', 'wallet'].edge_index = edge_txn_to_wallet
data['wallet', 'interacts_with', 'wallet'].edge_index = edge_wallet_to_wallet
data['transaction', 'flows_to', 'transaction'].edge_index = edge_txn_to_txn

print('=== HeteroData Object ===')
print(data)
print(f'\nNode types: {data.node_types}')
print(f'Edge types: {data.edge_types}')
print(f'\nTotal nodes: {data.num_nodes}')
print(f'Total edges: {data.num_edges}')

## 9. Train / Test Split

Following the anchor paper (Johannessen & Jullum):  
- **80/20 stratified split** on labeled nodes only  
- Unlabeled nodes remain in graph for message passing (semi-supervised)  
- **Primary task:** classify wallet nodes (accounts) as illicit/licit  
- Preserve natural class imbalance (no rebalancing)

In [ ]:
# ============================================================
# 9.1 Create masks for WALLET nodes (primary task)
# ============================================================

# Identify labeled wallet indices
labeled_wallet_mask = wallet_y >= 0
labeled_wallet_indices = torch.where(labeled_wallet_mask)[0].numpy()
labeled_wallet_labels = wallet_y[labeled_wallet_indices].numpy()

print(f'Total wallet nodes: {len(wallet_y)}')
print(f'Labeled wallet nodes: {len(labeled_wallet_indices)}')
print(f'Unlabeled wallet nodes: {(~labeled_wallet_mask).sum().item()}')

# Stratified 80/20 split
train_idx, test_idx = train_test_split(
    labeled_wallet_indices,
    test_size=0.2,
    stratify=labeled_wallet_labels,
    random_state=42
)

# Create boolean masks
num_wallets = len(wallet_y)
train_mask = torch.zeros(num_wallets, dtype=torch.bool)
test_mask = torch.zeros(num_wallets, dtype=torch.bool)
train_mask[train_idx] = True
test_mask[test_idx] = True

data['wallet'].train_mask = train_mask
data['wallet'].test_mask = test_mask

# Verify split
train_labels = wallet_y[train_mask]
test_labels = wallet_y[test_mask]

print(f'\n--- Train Set ---')
print(f'Total: {train_mask.sum().item()}')
print(f'Illicit: {(train_labels == 1).sum().item()}')
print(f'Licit:   {(train_labels == 0).sum().item()}')
print(f'Illicit ratio: {(train_labels == 1).sum().item() / train_mask.sum().item() * 100:.2f}%')

print(f'\n--- Test Set ---')
print(f'Total: {test_mask.sum().item()}')
print(f'Illicit: {(test_labels == 1).sum().item()}')
print(f'Licit:   {(test_labels == 0).sum().item()}')
print(f'Illicit ratio: {(test_labels == 1).sum().item() / test_mask.sum().item() * 100:.2f}%')

In [ ]:
# ============================================================
# 9.2 Also create masks for TRANSACTION nodes (secondary task)
# ============================================================

labeled_txn_mask = txn_y >= 0
labeled_txn_indices = torch.where(labeled_txn_mask)[0].numpy()
labeled_txn_labels = txn_y[labeled_txn_indices].numpy()

train_txn_idx, test_txn_idx = train_test_split(
    labeled_txn_indices,
    test_size=0.2,
    stratify=labeled_txn_labels,
    random_state=42
)

num_txns = len(txn_y)
train_txn_mask = torch.zeros(num_txns, dtype=torch.bool)
test_txn_mask = torch.zeros(num_txns, dtype=torch.bool)
train_txn_mask[train_txn_idx] = True
test_txn_mask[test_txn_idx] = True

data['transaction'].train_mask = train_txn_mask
data['transaction'].test_mask = test_txn_mask

print(f'Transaction train set: {train_txn_mask.sum().item()} '
      f'(illicit: {(txn_y[train_txn_mask] == 1).sum().item()})')
print(f'Transaction test set:  {test_txn_mask.sum().item()} '
      f'(illicit: {(txn_y[test_txn_mask] == 1).sum().item()})')

## 10. Dataset Summary & Comparison with Anchor Paper

In [ ]:
# ============================================================
# 10.1 Print final dataset summary
# ============================================================

print('=' * 65)
print('       HETEROGENEOUS GRAPH DATASET SUMMARY')
print('=' * 65)

print(f'\n{"NODE TYPES":=^50}')
print(f'{"Type":<20} {"#Nodes":>10} {"#Features":>10} {"#Labeled":>10}')
print('-' * 50)
print(f'{"Wallet (Account)":<20} {wallet_x.shape[0]:>10,} {wallet_x.shape[1]:>10} '
      f'{(wallet_y >= 0).sum().item():>10,}')
print(f'{"Transaction":<20} {txn_x.shape[0]:>10,} {txn_x.shape[1]:>10} '
      f'{(txn_y >= 0).sum().item():>10,}')
print(f'{"TOTAL":<20} {wallet_x.shape[0] + txn_x.shape[0]:>10,}')

print(f'\n{"EDGE TYPES (META-STEPS)":=^50}')
print(f'{"Type":<35} {"#Edges":>10}')
print('-' * 50)
print(f'{"wallet -> txn (sends)":<35} {edge_wallet_to_txn.shape[1]:>10,}')
print(f'{"txn -> wallet (received_by)":<35} {edge_txn_to_wallet.shape[1]:>10,}')
print(f'{"wallet -> wallet (interacts_with)":<35} {edge_wallet_to_wallet.shape[1]:>10,}')
print(f'{"txn -> txn (flows_to)":<35} {edge_txn_to_txn.shape[1]:>10,}')
print(f'{"TOTAL":<35} {total_edges:>10,}')

print(f'\n{"CLASS DISTRIBUTION (WALLETS)":=^50}')
print(f'{"Set":<15} {"Illicit":>10} {"Licit":>10} {"Ratio":>10}')
print('-' * 50)
n_train_ill = (train_labels == 1).sum().item()
n_train_lic = (train_labels == 0).sum().item()
n_test_ill = (test_labels == 1).sum().item()
n_test_lic = (test_labels == 0).sum().item()
print(f'{"Train (80%)":<15} {n_train_ill:>10,} {n_train_lic:>10,} '
      f'{n_train_ill/(n_train_ill+n_train_lic)*100:>9.2f}%')
print(f'{"Test  (20%)":<15} {n_test_ill:>10,} {n_test_lic:>10,} '
      f'{n_test_ill/(n_test_ill+n_test_lic)*100:>9.2f}%')

print('\n' + '=' * 65)

In [ ]:
# ============================================================
# 10.2 Comparison table: Our dataset vs Anchor paper
# ============================================================

print(f'\n{"COMPARISON: Our Study vs Anchor Paper":=^65}')
print(f'{"":<25} {"Ours (Elliptic++)":>20} {"Anchor (DNB)":>18}')
print('-' * 65)
print(f'{"Dataset":<25} {"Public":>20} {"Private":>18}')
print(f'{"Domain":<25} {"Bitcoin":>20} {"Banking":>18}')
print(f'{"Node types":<25} {"2":>20} {"3":>18}')
print(f'{"Edge types":<25} {"4":>20} {"2 (9 meta-steps)":>18}')
n_nodes = wallet_x.shape[0] + txn_x.shape[0]
print(f'{"Total nodes":<25} {n_nodes:>20,} {"5,149,159":>18}')
print(f'{"Total edges":<25} {total_edges:>20,} {"10,202,950":>18}')
print(f'{"Task":<25} {"Wallet classif.":>20} {"Individual classif.":>18}')
print(f'{"Train/Test split":<25} {"80/20 stratified":>20} {"80/20 stratified":>18}')
print(f'{"Class rebalancing":<25} {"No":>20} {"No":>18}')
print('-' * 65)

## 11. Save Preprocessed Data

In [ ]:
# ============================================================
# Save the HeteroData object for use in training notebook
# ============================================================
import os

SAVE_PATH = '/content/drive/MyDrive/EllipticPlusPlus/preprocessed'
os.makedirs(SAVE_PATH, exist_ok=True)

save_file = os.path.join(SAVE_PATH, 'hetero_graph_data.pt')
torch.save(data, save_file)
print(f'Saved preprocessed HeteroData to: {save_file}')

# Also save scalers for reproducibility
import pickle
scalers = {'txn_scaler': scaler_txn, 'wallet_scaler': scaler_wallet}
with open(os.path.join(SAVE_PATH, 'scalers.pkl'), 'wb') as f:
    pickle.dump(scalers, f)
print(f'Saved scalers to: {SAVE_PATH}/scalers.pkl')

print('\nDone! The preprocessed data is ready for model training.')

## 12. Quick Validation: Load & Verify

In [ ]:
# Quick check: reload and verify
data_loaded = torch.load(save_file)
print('Reloaded HeteroData:')
print(data_loaded)
print(f'\nWallet train nodes: {data_loaded["wallet"].train_mask.sum().item()}')
print(f'Wallet test nodes:  {data_loaded["wallet"].test_mask.sum().item()}')
print(f'\nReady for model training!')

---

## Next Steps

The preprocessed `HeteroData` object is saved and ready. In the next notebook, we will:

1. **Implement HMPNN** (Heterogeneous Message Passing Neural Network) following Algorithm 1 from Johannessen & Jullum
2. **Implement baselines**: Logistic Regression, XGBoost, GCN (homogeneous)
3. **Train & evaluate** using PR-AUC, ROC-AUC, Precision@Recall
4. **Compare results** across methods